In [10]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
usuario = os.getenv("BCCH_USER")
password = os.getenv("BCCH_PASS")

url = "https://si3.bcentral.cl/SieteRestWS/SieteRestWS.ashx"

params = {
    "user": usuario,
    "pass": password,
    "function": "GetSeries",
    "timeseries": "G073.IPC.IND.2023.M",
    "firstdate": "2015-01-01",
    "lastdate": "2025-12-31",
}

respuesta = requests.get(url, params=params, timeout=30)
datos = respuesta.json()

print("seriesId confirmado:", datos["Series"]["seriesId"])

observaciones = datos["Series"]["Obs"]
df_ipc = pd.DataFrame(observaciones)
df_ipc["fecha"] = pd.to_datetime(df_ipc["indexDateString"], format="%d-%m-%Y")
df_ipc["valor"] = df_ipc["value"].astype(float)
df_ipc = df_ipc[["fecha", "valor"]].sort_values("fecha").reset_index(drop=True)

assert df_ipc.shape[0] > 120, f"Esperaba >120 filas, llegaron {df_ipc.shape[0]} - ¿serie congelada?"
assert df_ipc["fecha"].max().year >= 2025, "La serie no llega a 2025 - revisa el seriesId"

df_ipc.to_csv("../data/raw/ipc_raw.csv", index=False)
print("Guardado:", df_ipc.shape, "hasta", df_ipc["fecha"].max().date())

seriesId confirmado: G073.IPC.IND.2023.M
Guardado: (132, 2) hasta 2025-12-01


In [11]:
params_busqueda_uf = { #Crea la busqueda
    "user": usuario,
    "pass": password,
    "function": "SearchSeries",
    "frequency": "DAILY"
}

respuesta_busqueda_uf = requests.get(url, params=params_busqueda_uf, timeout=30) #Ejecuta la busqueda

print("Status:", respuesta_busqueda_uf.status_code)
print(respuesta_busqueda_uf.text[:300])

Status: 200
{
  "Codigo": 0,
  "Descripcion": "Success",
  "Series": {
    "descripEsp": null,
    "descripIng": null,
    "seriesId": null,
    "Obs": null
  },
  "SeriesInfos": [
    {
      "seriesId": "G073.TCMX.IND.199801.D",
      "frequencyCode": "DAILY",
      "spanishTitle": "Índice TCM-X 


In [12]:
datos_busqueda = respuesta_busqueda_uf.json()
series_diarias = datos_busqueda["SeriesInfos"]

In [13]:
candidatos_uf = [
    s for s in series_diarias
    if "unidad de fomento" in s["spanishTitle"].lower()
]

print("Coincidencias:", len(candidatos_uf))
for s in candidatos_uf:
    print(s["seriesId"], "|", s["spanishTitle"], "|", s["lastObservation"])

Coincidencias: 1
F073.UFF.PRE.Z.D | Unidad de fomento (UF) | 09-07-2026


In [14]:
params_uf = {
    "user": usuario,
    "pass": password,
    "function": "GetSeries",
    "timeseries": "F073.UFF.PRE.Z.D",
    "firstdate": "2015-01-01",
    "lastdate": "2025-12-31",
}

respuesta_uf = requests.get(url, params_uf, timeout=30)
print(respuesta_uf.status_code)
print(respuesta_uf.text[:300])

200
{
  "Codigo": 0,
  "Descripcion": "Success",
  "Series": {
    "descripEsp": "Unidad de fomento (UF)",
    "descripIng": "Unit of account (UF)",
    "seriesId": "F073.UFF.PRE.Z.D",
    "Obs": [
      {
        "indexDateString": "01-01-2015",
        "value": "24627.1",
        "statusCod


In [15]:
datos_uf = respuesta_uf.json()
observaciones_uf = datos_uf["Series"]["Obs"]
df_uf = pd.DataFrame(observaciones_uf)

df_uf["fecha"] = pd.to_datetime(df_uf["indexDateString"], format="%d-%m-%Y")
df_uf["valor"] = df_uf["value"].astype(float)

df_uf = df_uf[["fecha", "valor"]]
df_uf = df_uf.sort_values("fecha").reset_index(drop=True)

print(df_uf.shape)
print("Fecha mínima:", df_uf["fecha"].min())
print("Fecha máxima:", df_uf["fecha"].max())
df_uf.sample(10)

(4018, 2)
Fecha mínima: 2015-01-01 00:00:00
Fecha máxima: 2025-12-31 00:00:00


,fecha,valor
2508,2021-11-13,30538.47
286,2015-10-14,25420.42
3412,2024-05-05,37286.78
1361,2018-09-23,27344.69
2174,2020-12-14,29086.29
3555,2024-09-25,37891.50
680,2016-11-11,26280.25
602,2016-08-25,26198.96
895,2017-06-14,26650.88
3953,2025-10-28,39582.38


In [16]:
df_uf.to_csv("../data/raw/uf_raw.csv", index=False)

print("Guardado:", df_uf.shape)

Guardado: (4018, 2)
